In [1]:
from transformers import pipeline

# ---------- Sentiment Analysis ---------
sentiment_analyzer = pipeline("sentiment-analysis")

reviews = [
    "The new smartphone has an amazing camera and battery life!",
    "The delivery was late and the packaging was damaged.",
]

for review in reviews:
    result = sentiment_analyzer(review)[0]
    print(f"Review: {review}\n-> {result['label']} ({round(result['score'], 3)})\n")

# ---------- Document Classification (Zero-Shot) ---------
classifier = pipeline(
    "zero-shot-classification", model="facebook/bart-large-mnli"
)

document = "The central bank raised interest rates to control rising inflation."
candidate_labels = ["Politics", "Economy", "Sports", "Technology"]

classification = classifier(document, candidate_labels)

print("Document:", document)
for label, score in zip(classification["labels"], classification["scores"]):
    print(f"{label}: {round(score, 3)}")

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Document: The central bank raised interest rates to control rising inflation.
Economy: 0.678
Politics: 0.126
Technology: 0.121
Sports: 0.075


In [5]:
from sentence_transformers import SentenceTransformer, util
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# 1. Knowledge Base
documents = [
    "The Eiffel Tower is located in Paris, France and was completed in 1889.",
    "Retrieval-Augmented Generation combines document retrieval with text generation.",
    "Python is a popular high-level programming language used in AI development.",
    "Vector databases store embeddings and support fast similarity search.",
]

# 2. Embed Documents as PyTorch Tensors
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
doc_embeddings = embed_model.encode(documents, convert_to_tensor=True)

# 3. Query and Retrieve Top-2 Relevant Chunks
query = "What is RAG in AI?"
query_embedding = embed_model.encode(query, convert_to_tensor=True)

# Use built-in cosine similarity search
hits = util.semantic_search(query_embedding, doc_embeddings, top_k=2)[0]
retrieved_chunks = [documents[hit['corpus_id']] for hit in hits]

# 4. Build Augmented Prompt
context = " ".join(retrieved_chunks)
prompt = f"Context: {context}\nQuestion: {query}\nAnswer:"

# 5. Generate Answer (Bypassing pipeline entirely)
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
llm_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

# Tokenize the input prompt
inputs = tokenizer(prompt, return_tensors="pt")

# Generate the output tokens
outputs = llm_model.generate(**inputs, max_new_tokens=60)

# Decode the tokens back into readable text
answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

# Output Results
print("Retrieved Context:")
for chunk in retrieved_chunks:
    print(f" - {chunk}")

print("\nGenerated Answer:")
print(answer)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Retrieved Context:
 - Python is a popular high-level programming language used in AI development.
 - Retrieval-Augmented Generation combines document retrieval with text generation.

Generated Answer:
combines document retrieval with text generation


In [6]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

tokenizer = AutoTokenizer.from_pretrained("Salesforce/codegen-350M-mono")
model = AutoModelForCausalLM.from_pretrained("Salesforce/codegen-350M-mono")

def generate_code(prompt, max_new_tokens=80):
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids
    output = model.generate(
        input_ids,
        max_new_tokens=max_new_tokens,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=False
    )
    return tokenizer.decode(output[0], skip_special_tokens=True)

# 1. Code generation from a natural-language instruction
prompt1 = "# Write a Python function to check if a number is prime\ndef is_prime(n):"
print("Generated Function:\n", generate_code(prompt1))

# 2. Debugging a faulty snippet
buggy_code = """# The following function should return the factorial of n, but has a bug. Fix it.
def factorial(n):
    result = 0
    for i in range(1, n+1):
        result = result * i
    return result

# Corrected function:
def factorial_fixed(n):"""

print("\nDebug Suggestion:\n", generate_code(buggy_code, max_new_tokens=60))

config.json:   0%|          | 0.00/999 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/240 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/1.00k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/165 [00:00<?, ?it/s]

[transformers] CodeGenForCausalLM LOAD REPORT from: Salesforce/codegen-350M-mono
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...19}.attn.causal_mask | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors: reconstructing file:   0%|          |  0.00B /  797MB            

model.safetensors: downloading bytes:           |  0.00B            

Generated Function:
 # Write a Python function to check if a number is prime
def is_prime(n):
    if n == 2 or n == 3:
        return True
    if n % 2 == 0 or n % 3 == 0:
        return False
    for i in range(5, int(math.sqrt(n)) + 1, 6):
        if n % i == 0:
            return False
    return True


# Write a Python

Debug Suggestion:
 # The following function should return the factorial of n, but has a bug. Fix it.
def factorial(n):
    result = 0
    for i in range(1, n+1):
        result = result * i
    return result

# Corrected function:
def factorial_fixed(n):
    result = 1
    for i in range(1, n+1):
        result = result * i
    return result

# The following function should return the factorial of n, but has a bug. Fix it.
def factorial_fixed2(n


In [8]:
from diffusers import StableDiffusionPipeline
import torch

# Removed torch_dtype=torch.float16 because float32 is better supported on CPU
pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5"
)

# Removed the .to("cuda") line so it defaults to CPU

prompt = "A futuristic city skyline at sunset, digital art, highly detailed"
image = pipe(
    prompt,
    num_inference_steps=30,
    guidance_scale=7.5
).images[0]

image.save("generated_city.png")
print("Image generated and saved as generated_city.png")

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

Image generated and saved as generated_city.png
